In [0]:
from pyspark.sql.functions import col, lower

access_package_apps_df = spark.table(
    "adani_group_enterprise.identitygovernance.access_package_apps"
)

access_package_groups_df = spark.table(
    "adani_group_enterprise.identitygovernance.access_package_groups"
)

databricks_agent_apps_df = (
    access_package_apps_df.alias("apps")
    .join(
        access_package_groups_df.alias("groups"),
        col("apps.accessPackageId") == col("groups.accessPackageId"),
        "inner"
    )
    .filter(
        (lower(col("apps.platformType")) == "databricks") &
        (lower(col("apps.appType")) == "agent") #&
#        (col("apps.is_active") == True) &
#        (col("groups.is_active") == True)
    )
    .select(
        col("apps.accessPackageId"),
        col("apps.appId"),
        col("apps.appName"),
        col("apps.roleId"),
        col("apps.roleName"),
        col("apps.appType"),
        col("apps.cloudType"),
        col("apps.platformType"),
        col("apps.targetAppId"),
        col("apps.targetAppUrl"),
        col("apps.is_active").alias("app_is_active"),
        col("groups.groupId"),
        col("groups.groupName"),
        col("groups.is_active").alias("group_is_active")
    )
)

databricks_agent_apps_df.show(truncate=False)

+------------------------------------+------------------------------------+-------------------+------------------------------------+--------+-------+---------+------------+-------------+------------------------------------------+-------------+------------------------------------+------------------------+---------------+
|accessPackageId                     |appId                               |appName            |roleId                              |roleName|appType|cloudType|platformType|targetAppId  |targetAppUrl                              |app_is_active|groupId                             |groupName               |group_is_active|
+------------------------------------+------------------------------------+-------------------+------------------------------------+--------+-------+---------+------------+-------------+------------------------------------------+-------------+------------------------------------+------------------------+---------------+
|61478292-23f6-4fc5-8bf5-7b063fbba

In [0]:
import requests

def apply_databricks_app_permission(workspace_url, token, app_id, group_name, permission_level="CAN_MANAGE", group_is_active=True, app_is_active=True):
    
    url = f"https://{workspace_url}/api/2.0/permissions/apps/{app_id}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    # Step 1: Get existing ACL
    get_resp = requests.get(url, headers=headers)
    get_resp.raise_for_status()
    getjson = get_resp.json()

    existing_acl = getjson.get("access_control_list", [])

    # Step 2: Normalize → extract only required fields
    normalized_acl = []

    for entry in existing_acl:
        if "group_name" in entry:
            normalized_acl.append({
                "group_name": entry["group_name"],
                "permission_level": entry["all_permissions"][0]["permission_level"]
            })
        elif "user_name" in entry:
            normalized_acl.append({
                "user_name": entry["user_name"],
                "permission_level": entry["all_permissions"][0]["permission_level"]
            })

    # Step 3: Remove existing entry for this group (avoid duplicates)
    normalized_acl = [
        acl for acl in normalized_acl 
        if acl.get("group_name") != group_name
    ]
    print(f'\n\n normalized_acl ---> {normalized_acl}')

    # Step 4: Add new/updated group entry
    if(group_is_active == True and app_is_active == True):
        normalized_acl.append({
            "group_name": group_name,
            "permission_level": permission_level
        })
    print(f'\n\n normalized_acl_2 ---> {normalized_acl}')

    # Step 5: Build payload
    payload = {
        "access_control_list": normalized_acl
    }
    print(f'\n\npayload ---> {payload}')
    # Step 6: PUT
    patch_resp = requests.put(url, headers=headers, json=payload)
    # Step 6: PATCH
    patch_resp = requests.patch(url, headers=headers, json=payload)

    print("STATUS:", patch_resp.status_code)
    print("TEXT:", patch_resp.text)

    return patch_resp.json()

In [0]:
workspace_url = "adb-7405608465126087.7.azuredatabricks.net"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)

rows = databricks_agent_apps_df.collect()

for row in rows:
    app_id = row["targetAppId"]
    group_name = row["groupName"]
    group_is_active = row["group_is_active"]
    app_is_active = row["app_is_active"]

    if app_id and group_name:
        print(f"Applying permission for app_id={app_id}, group={group_name}, group_is_active={group_is_active}, app_is_active={app_is_active}")

        apply_databricks_app_permission(
            workspace_url=workspace_url,
            token=token,
            app_id=app_id,
            group_name=group_name,
            permission_level="CAN_MANAGE",
            group_is_active=group_is_active,
            app_is_active=app_is_active
        )
    else:
        print(f"Skipping row because app_id or group_name is missing: {row}")

Applying permission for app_id=abac-demo-app, group=ap_Cements_OneSales_Team, group_is_active=True, app_is_active=True


 normalized_acl ---> [{'group_name': 'admins', 'permission_level': 'CAN_MANAGE'}]


 normalized_acl_2 ---> [{'group_name': 'admins', 'permission_level': 'CAN_MANAGE'}, {'group_name': 'ap_Cements_OneSales_Team', 'permission_level': 'CAN_USE'}]


payload ---> {'access_control_list': [{'group_name': 'admins', 'permission_level': 'CAN_MANAGE'}, {'group_name': 'ap_Cements_OneSales_Team', 'permission_level': 'CAN_USE'}]}
STATUS: 200
TEXT: {"object_id":"/apps/b3f8b5ad-dbcd-4551-a21d-ac4d74efe2c6","object_type":"apps","access_control_list":[{"group_name":"admins","all_permissions":[{"permission_level":"CAN_MANAGE","inherited":false},{"permission_level":"CAN_MANAGE","inherited":true,"inherited_from_object":["/apps"]}]},{"group_name":"ap_Cements_OneSales_Team","all_permissions":[{"permission_level":"CAN_USE","inherited":false}]}]}
